# Sesión 21 — PySpark DataFrames II: agregaciones, ordenamiento y escritura de resultados

**Asignatura:** Big Data — Ciencia de Datos, Universidad Externado de Colombia  
**Tipo de actividad:** Presencial · Guiada en clase (formativa)  
**Objetivo:** Aplicar agrupaciones, agregaciones, ordenamientos y escritura de resultados sobre un dataset real con PySpark.

---

**Prerrequisitos:**
- Sesión 20: creación de la SparkSession sobre YARN, carga de datos desde HDFS, transformaciones (`select`, `filter`, `withColumn`, `drop`, `distinct`) y acciones fundamentales.

**Al finalizar esta sesión, el estudiante será capaz de:**
1. Realizar agrupaciones y agregaciones (`groupBy().agg()`) sobre un DataFrame.
2. Ordenar y limitar resultados (`orderBy()`, `limit()`).
3. Guardar resultados en formatos CSV y Parquet.

> **Nota:** El dataset (Global Power Plant Database) permanece disponible en el HDFS del clúster; no es necesario volver a descargarlo ni recargarlo.


In [6]:
# Se recrea la SparkSession y se vuelve a cargar el dataset desde HDFS
# (ya está disponible en el clúster; no es necesario volver a subirlo).
# También se importan 'col' y 'round', que se usarán en esta sesión.
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, round as spark_round

spark = SparkSession.builder \
    .appName("PySpark - Primeros pasos") \
    .master("yarn") \
    .getOrCreate()

df = spark.read.csv(
    "hdfs:///datasets/global_power_plant_database.csv",
    header=True,
    inferSchema=True
)

print("Versión de Spark:", spark.version)
print("Master:", spark.sparkContext.master)
print(f"El DataFrame tiene {df.count()} filas y {len(df.columns)} columnas.")

Versión de Spark: 3.5.7
Master: yarn
El DataFrame tiene 34936 filas y 36 columnas.


---

> **Nota:** No existe una única solución correcta para los ejercicios de esta sesión. Lo importante es que el resultado sea coherente y que se utilicen las operaciones de PySpark (no pandas).


### Ejercicio 1

**Evolución histórica:** Filtrar las plantas que tienen año de comisionamiento (`commissioning_year`) conocido (no nulo). Crear una nueva columna llamada `decada` que indique la década de puesta en operación (por ejemplo, 1990 para plantas comisionadas entre 1990 y 1999). Luego, agrupar por `decada` y `primary_fuel`, calcular el número de plantas en cada grupo y mostrar los 20 registros con mayor cantidad de plantas.

*Pista: para calcular la década se puede dividir el año entre 10, convertir a entero y multiplicar por 10.*

In [17]:
from pyspark.sql.functions import sum as spark_sum, count, avg, max as spark_max, min as spark_min

In [18]:
from pyspark.sql import functions as F
top_20 = (
    df.filter(F.col("commissioning_year").isNotNull())
    .withColumn("decada", (F.floor(F.col("commissioning_year") / 10) * 10).cast("int"))
    .groupBy("decada", "primary_fuel")
    .agg(F.count("*").alias("cantidad_plantas"))
    .orderBy(F.col("cantidad_plantas").desc())
    .limit(20)
)

top_20.show(20)

+------+------------+----------------+
|decada|primary_fuel|cantidad_plantas|
+------+------------+----------------+
|  2010|       Solar|            3313|
|  2010|        Wind|            1255|
|  2000|         Gas|             913|
|  2000|        Wind|             884|
|  2010|         Gas|             561|
|  2010|        Coal|             557|
|  2000|        Coal|             556|
|  1980|       Hydro|             539|
|  2010|       Hydro|             523|
|  1990|         Gas|             492|
|  2000|         Oil|             487|
|  2000|       Hydro|             464|
|  1960|       Hydro|             403|
|  2010|         Oil|             393|
|  1950|       Hydro|             390|
|  1990|       Hydro|             366|
|  1980|         Gas|             283|
|  1970|       Hydro|             283|
|  1920|       Hydro|             277|
|  1990|         Oil|             244|
+------+------------+----------------+



## 6. Agrupaciones y agregaciones

Al igual que en pandas (`df.groupby("col").agg(...)`), PySpark permite agrupar datos por una o más columnas y calcular funciones de agregación. La sintaxis es muy similar.

### Pregunta analítica 1: ¿Cuál es la capacidad total instalada por tipo de combustible a nivel mundial?

In [20]:
# Capacidad total por tipo de combustible
capacidad_por_fuel = (
    df.groupBy("primary_fuel")
    .agg(
        spark_sum("capacity_mw").alias("capacidad_total_mw"),
        count("*").alias("num_plantas")
    )
    .orderBy(col("capacidad_total_mw").desc())
)

capacidad_por_fuel.show(truncate=False)

+--------------+------------------+-----------+
|primary_fuel  |capacidad_total_mw|num_plantas|
+--------------+------------------+-----------+
|Coal          |1965540.9996000004|2330       |
|Gas           |1493050.6021489995|3998       |
|Hydro         |1053159.6197999984|7156       |
|Nuclear       |407911.75999999995|195        |
|Wind          |263053.7294999998 |5344       |
|Oil           |261878.7096180001 |2320       |
|Solar         |188312.32348999952|10665      |
|Biomass       |34281.30159999999 |1430       |
|Waste         |14748.713999999994|1068       |
|Geothermal    |12687.75          |189        |
|Cogeneration  |4048.0            |41         |
|Other         |3612.8599999999997|43         |
|Petcoke       |2424.5774999999994|12         |
|Storage       |1712.3000000000002|135        |
|Wave and Tidal|552.2             |10         |
+--------------+------------------+-----------+



In [23]:
# Extrae el número como variable de Python
Total_mw = capacidad_por_fuel.select(spark_sum("capacidad_total_mw")).first()[0]

print(f"Total global: {Total_mw:,.2f} MW")

Total global: 5,706,975.45 MW


### Pregunta analítica 2: ¿Cuáles son los 10 países con mayor número de centrales eléctricas registradas?

In [24]:
# Top 10 países por número de plantas
top_paises = (
    df.groupBy("country_long")
    .agg(count("*").alias("num_plantas"))
    .orderBy(col("num_plantas").desc())
    .limit(10)
)

top_paises.show(truncate=False)

+------------------------+-----------+
|country_long            |num_plantas|
+------------------------+-----------+
|United States of America|9833       |
|China                   |4235       |
|United Kingdom          |2751       |
|Brazil                  |2360       |
|France                  |2155       |
|India                   |1589       |
|Germany                 |1309       |
|Canada                  |1159       |
|Spain                   |829        |
|Russia                  |545        |
+------------------------+-----------+



### Pregunta analítica 3: ¿Cuál es la capacidad promedio, máxima y mínima por tipo de combustible?

In [25]:
# Capacidad promedio, máxima y mínima por tipo de combustible
stats_fuel = (
    df.groupBy("primary_fuel")
    .agg(
        spark_round(avg("capacity_mw"), 2).alias("capacidad_promedio_mw"),
        spark_max("capacity_mw").alias("capacidad_maxima_mw"),
        spark_min("capacity_mw").alias("capacidad_minima_mw")
    )
    .orderBy(col("capacidad_promedio_mw").desc())
)

stats_fuel.show(truncate=False)

+--------------+---------------------+-------------------+-------------------+
|primary_fuel  |capacidad_promedio_mw|capacidad_maxima_mw|capacidad_minima_mw|
+--------------+---------------------+-------------------+-------------------+
|Nuclear       |2091.86              |8212.0             |20.0               |
|Coal          |843.58               |7000.0             |1.2                |
|Gas           |373.45               |8865.0             |1.0                |
|Petcoke       |202.05               |1707.3             |4.7                |
|Hydro         |147.17               |22500.0            |1.0                |
|Oil           |112.88               |6794.0             |1.0                |
|Cogeneration  |98.73                |1404.0             |1.6                |
|Other         |84.02                |845.26             |4.5                |
|Geothermal    |67.13                |1163.0             |1.0                |
|Wave and Tidal|55.22                |254.0         

### Ejercicio 2

**Análisis energético de Colombia:** Filtrar las plantas ubicadas en Colombia (`country == "COL"`) y calcular la capacidad total instalada agrupada por tipo de combustible (`primary_fuel`). Ordenar el resultado de mayor a menor capacidad. ¿Cuál es la fuente de energía dominante en Colombia?

In [39]:
# Escribir el código aquí
capacidad_colombia = (
    df.filter(F.col("country") == "COL")
    .groupBy("primary_fuel")
    .agg(
        F.round(F.sum("capacity_mw"), 2).alias("capacidad_total_mw"),
        F.count("*").alias("num_plantas")
    )
    .orderBy(F.col("capacidad_total_mw").desc())
)

capacidad_colombia.show(truncate=False)

+------------+------------------+-----------+
|primary_fuel|capacidad_total_mw|num_plantas|
+------------+------------------+-----------+
|Hydro       |6694.0            |12         |
|Gas         |2553.0            |5          |
|Coal        |1393.0            |5          |
|Oil         |188.0             |1          |
|Solar       |96.0              |2          |
+------------+------------------+-----------+



### Ejercicio 3

**Comparación renovable vs. no renovable:** Crear una nueva columna llamada `tipo_energia` que clasifique cada planta como `"Renovable"` o `"No renovable"` según su `primary_fuel`. Se consideran **renovables**: Solar, Wind, Hydro, Biomass, Geothermal, Wave and Tidal. El resto se clasifica como No renovable.

Luego, agrupar por `tipo_energia` y calcular: (a) el número total de plantas, (b) la capacidad total en MW, y (c) la capacidad promedio por planta.

*Pista: se puede usar `when().otherwise()` de `pyspark.sql.functions` para crear la clasificación condicional.*

In [29]:
# Pista: importar 'when'
from pyspark.sql.functions import when

# Escribir el código aquí

# 1. Definir la lista de fuentes renovables
renovables = ["Solar", "Wind", "Hydro", "Biomass", "Geothermal", "Wave and Tidal"]

# 2. Crear la columna condicional con when().otherwise()
df_clasificado = df.withColumn(
    "tipo_energia",
    F.when(F.col("primary_fuel").isin(renovables), "Renovable").otherwise("No renovable")
)

# 3. Agrupar por 'tipo_energia' y calcular las métricas solicitadas
resumen_energia = (
    df_clasificado.groupBy("tipo_energia")
    .agg(
        F.count("*").alias("total_plantas"),
        F.round(F.sum("capacity_mw"), 2).alias("capacidad_total_mw"),
        F.round(F.avg("capacity_mw"), 2).alias("capacidad_promedio_mw")
    )
    .orderBy(F.col("capacidad_total_mw").desc())
)

resumen_energia.show(truncate=False)

+------------+-------------+------------------+---------------------+
|tipo_energia|total_plantas|capacidad_total_mw|capacidad_promedio_mw|
+------------+-------------+------------------+---------------------+
|No renovable|10142        |4154928.52        |409.68               |
|Renovable   |24794        |1552046.92        |62.6                 |
+------------+-------------+------------------+---------------------+



## 7. Ordenamiento y limitación

- `orderBy()` / `sort()`: equivalentes a `df.sort_values()` en pandas. Son sinónimos en PySpark.
- `limit(n)`: equivalente a `df.head(n)` en pandas, pero retorna un DataFrame (no un valor escalar).

La combinación de agrupación + ordenamiento + limitación permite responder preguntas del tipo *"los N más grandes/pequeños según..."*.

In [30]:
# Las 10 centrales eléctricas más grandes del mundo
top_10_plantas = (
    df.select("name", "country_long", "primary_fuel", "capacity_mw")
    .orderBy(col("capacity_mw").desc())
    .limit(10)
)

top_10_plantas.show(truncate=False)

+---------------------------------------+------------+------------+-----------+
|name                                   |country_long|primary_fuel|capacity_mw|
+---------------------------------------+------------+------------+-----------+
|Three Gorges Dam                       |China       |Hydro       |22500.0    |
|Baihetan Dam                           |China       |Hydro       |13050.0    |
|Xiluodu                                |China       |Hydro       |12600.0    |
|Surgutskaya GRES-2                     |Russia      |Gas         |8865.0     |
|Simon Bolivar (Guri)                   |Venezuela   |Hydro       |8851.0     |
|Tucuruí                                |Brazil      |Hydro       |8535.0     |
|Kashiwazaki Kariwa                     |Japan       |Nuclear     |8212.0     |
|Itaipu (Parte Brasileira)              |Brazil      |Hydro       |7000.0     |
|East Hope Metals Wucaiwan power station|China       |Coal        |7000.0     |
|Itaipu Binacional Dam (Paraguay part)  

In [31]:
# Las 10 centrales más antiguas (con año de comisionamiento conocido)
mas_antiguas = (
    df.filter(col("commissioning_year").isNotNull())
    .select("name", "country_long", "primary_fuel", "capacity_mw", "commissioning_year")
    .orderBy("commissioning_year")
    .limit(10)
)

mas_antiguas.show(truncate=False)

+-----------------------+------------------------+------------+------------------+------------------+
|name                   |country_long            |primary_fuel|capacity_mw       |commissioning_year|
+-----------------------+------------------------+------------+------------------+------------------+
|Granite                |United States of America|Hydro       |2.0               |1896.0            |
|Santa Ana 1            |United States of America|Hydro       |3.2               |1899.0            |
|Gurtnellen             |Switzerland             |Hydro       |6.6               |1900.0            |
|Centerville PH         |United States of America|Hydro       |6.4               |1900.0            |
|Telesp                 |Brazil                  |Oil         |1.12              |1900.0            |
|Samara GRES            |Russia                  |Gas         |61.0              |1900.0            |
|Bompreço Fortaleza     |Brazil                  |Oil         |1.12              |

## 8. Escritura de resultados

Una vez realizadas las transformaciones y análisis, los resultados se pueden guardar en diferentes formatos. Los dos más comunes son CSV (legible por humanos y otras herramientas) y Parquet (formato columnar optimizado para Big Data).

### 8.1. Guardar en formato CSV

In [32]:
# Ver el esquema del DataFrame antes de guardarlo
top_paises

DataFrame[country_long: string, num_plantas: bigint]

In [33]:
# Guardar el top 10 de países en CSV
top_paises.write.csv("/top_10_paises_csv", header=True, mode="overwrite")

# Verificar los archivos generados
!hdfs dfs -ls /top_10_paises_csv/

Found 2 items
-rw-r--r--   2 hadoop supergroup          0 2026-09-16 17:33 /top_10_paises_csv/_SUCCESS
-rw-r--r--   2 hadoop supergroup        167 2026-09-16 17:33 /top_10_paises_csv/part-00000-bbb04007-71ee-4abb-8cc2-e4e7a8827a48-c000.csv


**Observación:** Spark no genera un único archivo CSV, sino un directorio con múltiples archivos `part-*.csv`. Esto ocurre porque Spark está diseñado para escritura **paralela**: cada executor escribe su propia partición. En un clúster real, este comportamiento permite guardar terabytes de datos de forma eficiente.

Si se necesita un único archivo (por ejemplo, para análisis local), se puede usar `.coalesce(1)` antes de `.write` para forzar una sola partición:

In [34]:
# Forzar una sola partición con coalesce(1) para obtener un único archivo part-*.csv
# Nota: el destino sigue siendo un directorio, no un archivo .csv suelto
top_paises.coalesce(1).write.csv("top_10_paises_csv_single", header=True, mode="overwrite")

!hdfs dfs -ls top_10_paises_csv_single/

Found 2 items
-rw-r--r--   2 hadoop supergroup          0 2026-09-16 17:36 top_10_paises_csv_single/_SUCCESS
-rw-r--r--   2 hadoop supergroup        167 2026-09-16 17:36 top_10_paises_csv_single/part-00000-95efadae-a92e-48ab-88dc-2ec1fd82af9c-c000.csv


### 8.2. Guardar en formato Parquet

In [35]:
# Guardar la capacidad por tipo de combustible en Parquet
capacidad_por_fuel.write.parquet("/capacidad_por_fuel_parquet", mode="overwrite")

!hdfs dfs -ls /capacidad_por_fuel_parquet/

Found 2 items
-rw-r--r--   2 hadoop supergroup          0 2026-09-16 17:36 /capacidad_por_fuel_parquet/_SUCCESS
-rw-r--r--   2 hadoop supergroup       1386 2026-09-16 17:36 /capacidad_por_fuel_parquet/part-00000-459d8c45-418c-4a64-8bf0-23d04149e40e-c000.snappy.parquet


### ¿Por qué Parquet es relevante en Big Data?

Parquet es un formato de almacenamiento **columnar** (a diferencia de CSV, que es basado en filas). Esto significa que los datos de cada columna se almacenan juntos en disco. Las principales ventajas son:

- **Compresión eficiente:** los valores de una misma columna tienden a ser similares, lo que permite mejor compresión.
- **Lectura selectiva:** si una consulta solo necesita 3 de 36 columnas, Spark puede leer únicamente esas 3, ahorrando I/O (Entrada/Salida).
- **Preservación de tipos:** a diferencia de CSV, Parquet almacena el esquema dentro del archivo. No es necesario inferir tipos al leer.

En sesiones posteriores se profundizará en cómo Spark aprovecha Parquet junto con técnicas como el *predicate pushdown* y la *partition pruning*.

In [36]:
# Lectura del archivo Parquet recién creado (verificación)
df_parquet = spark.read.parquet("/capacidad_por_fuel_parquet")
df_parquet.show(truncate=False)

+--------------+------------------+-----------+
|primary_fuel  |capacidad_total_mw|num_plantas|
+--------------+------------------+-----------+
|Coal          |1965540.9996000004|2330       |
|Gas           |1493050.6021489995|3998       |
|Hydro         |1053159.6197999984|7156       |
|Nuclear       |407911.75999999995|195        |
|Wind          |263053.7294999998 |5344       |
|Oil           |261878.7096180001 |2320       |
|Solar         |188312.32348999952|10665      |
|Biomass       |34281.30159999999 |1430       |
|Waste         |14748.713999999994|1068       |
|Geothermal    |12687.75          |189        |
|Cogeneration  |4048.0            |41         |
|Other         |3612.8599999999997|43         |
|Petcoke       |2424.5774999999994|12         |
|Storage       |1712.3000000000002|135        |
|Wave and Tidal|552.2             |10         |
+--------------+------------------+-----------+



### Ejercicio 4

**Exportación de resultados:** Tomando el resultado del Ejercicio 2 (capacidad por combustible en Colombia), guardar el DataFrame resultante en formato Parquet en un directorio llamado `"colombia_capacidad_parquet"`. Luego, leer el archivo Parquet recién creado con `spark.read.parquet()` y verificar que los datos se preservaron correctamente mostrando el contenido.

In [42]:
# Escribir el código aquí
colombia_capacidad_parquet = spark.read.parquet("/capacidad_colombia")
colombia_capacidad_parquet.show(truncate=False)

AnalysisException: [PATH_NOT_FOUND] Path does not exist: hdfs://hadoop-virtualbox:9000/capacidad_colombia.

---

## Cierre

En estas dos sesiones (20 y 21) se cubrieron las operaciones fundamentales de PySpark: creación de la SparkSession sobre el clúster, carga de datos desde HDFS, exploración, transformaciones, acciones, agrupaciones, ordenamientos y escritura de resultados.

En la próxima sesión (Sesión 22 — Taller N.º 6) se practicará lo aprendido directamente sobre el clúster de la sala Deep Learning, documentando tiempos de ejecución y decisiones de transformación tomadas. Spark SQL, el Catalyst Optimizer, Adaptive Query Execution y la lectura de planes de consulta se abordarán formalmente en la Sesión 23.

**Acción recomendada:**
- Completar los ejercicios pendientes, en caso de no haberlos terminado en clase.
- Experimentar libremente con el dataset: formular preguntas propias e intentar responderlas con las operaciones aprendidas.


In [43]:
# Detener la SparkSession al finalizar
spark.stop()